# Diffusion World Model on Kaggle (Push-T & Visual AntMaze)

This notebook clones your `diffusion-wm` GitHub repository, sets up a Python 3.11 `uv` environment, prepares the datasets and fixed evaluation tasks, trains the **SD-VAE + Rectified Flow / Diffusion Forcing DiT** world model, and evaluates closed-loop MPC planning (`GradientSolver` or `CEMSolver`).

> **Kaggle Settings Required:**
> - **Accelerator:** GPU (T4 x2 or P100 16 GB)
> - **Internet:** ON (to clone GitHub & download `stabilityai/sd-vae-ft-mse`, datasets, and eval tasks)

In [ ]:
# === 1. User Configuration ===
GITHUB_REPO_URL = "https://github.com/hanriman/diffusion-wm.git"
BRANCH = "main"

ENV_NAME = "pusht"   # "pusht" or "ant"
SOLVER = "gd"        # "gd" (GradientSolver) or "cem" (CEMSolver)
SEED = 42

PRECOMPUTE_VAE_LATENTS = True  # recommended on Kaggle 16GB GPU for faster training
MAX_EPOCHS = 10 if ENV_NAME == "pusht" else 5
BATCH_SIZE = 64 if PRECOMPUTE_VAE_LATENTS else (32 if ENV_NAME == "pusht" else 64)
NUM_EVAL = 50        # set lower (e.g. 5 or 10) for quick testing
EVAL_NUM_SAMPLES = 64 if SOLVER == "gd" else 128

# Store large HDF5 datasets in /tmp (ephemeral, high capacity) and checkpoints in /kaggle/working
REPO_DIR = "/kaggle/working/diffusion-wm"
DATA_DIR = "/tmp/hjepa_data"
CKPT_DIR = f"/kaggle/working/ckpts/{ENV_NAME}/{ENV_NAME}_dwm/seed{SEED}"

# Optional: if you uploaded datasets as a Kaggle Dataset (e.g. /kaggle/input/pusht-h5), set path here:
KAGGLE_INPUT_DATA_DIR = ""  # leave empty to download automatically from HuggingFace

In [ ]:
%%bash
nvidia-smi
df -h /kaggle/working /tmp
apt-get update -qq && apt-get install -y -qq zstd > /dev/null || true
pip install -q uv
which uv && uv --version

In [ ]:
import os
import subprocess

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["HJEPA_HOME"] = DATA_DIR
os.environ["MUJOCO_GL"] = "egl"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

%cd {REPO_DIR}

In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
uv venv --python 3.11
uv pip install -e ".[env]" pytest
uv run python -c "import torch; print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')"

### 2. Download Evaluation Tasks & Datasets
- Downloads the 50 fixed evaluation tasks into `diffusion_wm/assets/eval_trajs/`.
- Either links `.h5` files from `KAGGLE_INPUT_DATA_DIR` (if provided) or downloads them from HuggingFace into `DATA_DIR` (`/tmp/hjepa_data`).

In [ ]:
import glob
import subprocess

subprocess.run(["bash", "scripts/download_eval_trajs.sh"], check=True)

if KAGGLE_INPUT_DATA_DIR and os.path.isdir(KAGGLE_INPUT_DATA_DIR):
    for h5_file in glob.glob(os.path.join(KAGGLE_INPUT_DATA_DIR, "*.h5")):
        dst = os.path.join(DATA_DIR, os.path.basename(h5_file))
        if not os.path.exists(dst):
            os.symlink(h5_file, dst)
    print("Linked HDF5 files from Kaggle Input:", os.listdir(DATA_DIR))
else:
    subprocess.run(["bash", "scripts/download_datasets.sh", ENV_NAME], check=True)

### 3. (Optional) Precompute SD-VAE Latents
Pre-encodes `pixels` with `stabilityai/sd-vae-ft-mse` into a `vae_latents` column inside each `.h5` file so training does not run the VAE encoder every epoch.

In [ ]:
if PRECOMPUTE_VAE_LATENTS:
    if ENV_NAME == "pusht":
        h5_files = ["pusht_expert_train.h5", "pusht_expert_val.h5"]
    else:
        h5_files = ["visual_antmaze_medium_explore_stitch_train.h5", "visual_antmaze_medium_stitch_val.h5"]
    for fname in h5_files:
        fpath = os.path.join(DATA_DIR, fname)
        subprocess.run([
            "uv", "run", "python", "scripts/precompute_vae_latents.py",
            fpath, "--batch-size", "64"
        ], check=True)

### 4. Train Diffusion World Model
Uses `trainer.precision=16-mixed` and `trainer.devices=1` for Kaggle T4/P100 compatibility, and saves checkpoints + `normalizer.pt` to `/kaggle/working/ckpts/...`.

In [ ]:
train_cfg = f"{ENV_NAME}_dwm"
cmd = [
    "uv", "run", "python", "-m", "diffusion_wm.train",
    "--config-name", train_cfg,
    f"seed={SEED}",
    f"cache_dir={DATA_DIR}",
    f"run_dir={CKPT_DIR}",
    "trainer.precision=16-mixed",
    "trainer.devices=1",
    f"trainer.max_epochs={MAX_EPOCHS}",
    f"loader.batch_size={BATCH_SIZE}",
    "loader.num_workers=2",
    "planning_eval.enabled=false",
]

if PRECOMPUTE_VAE_LATENTS:
    if ENV_NAME == "pusht":
        cmd.append("data.dataset.keys_to_load=[vae_latents,action,proprio,state]")
    else:
        cmd.append("data.dataset.keys_to_load=[vae_latents,action,xy,qpos,qvel,proprio]")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

### 5. Evaluate Closed-Loop MPC Planning
Runs `diffusion_wm.eval` using the trained checkpoint on the 50 fixed evaluation tasks and prints `metrics.yaml`.

In [ ]:
ckpt_path = os.path.join(CKPT_DIR, f"{ENV_NAME}_dwm_object.ckpt")
eval_cfg = f"{ENV_NAME}_{SOLVER}"
eval_out_dir = f"eval_{SOLVER}"

eval_cmd = [
    "uv", "run", "python", "-m", "diffusion_wm.eval",
    "--config-name", eval_cfg,
    f"seed={SEED}",
    f"cache_dir={DATA_DIR}",
    f"policy={ckpt_path}",
    f"eval.num_eval={NUM_EVAL}",
    f"solver.num_samples={EVAL_NUM_SAMPLES}",
    f"output.dir={eval_out_dir}",
]
print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)

metrics_file = os.path.join(CKPT_DIR, eval_out_dir, "metrics.yaml")
with open(metrics_file) as f:
    print("\n=== Evaluation Metrics ===\n" + f.read())